In [11]:
import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset, Subset, DataLoader
import numpy as np

from dataset import create_dataloaders
from cnn_baseline_1d import CNNBaseline1D
from train import train_one_epoch
from evaluate import evaluate, evaluate_top_models_person_cv
from grid_search import run_grid_search, run_person_grid_search

In [12]:
BATCH_SIZE = 8

INTRA_DATA_DIR = "preprocessed_data/Intra"
intra_train_loader, intra_test_loader = create_dataloaders(INTRA_DATA_DIR, BATCH_SIZE, add_person_id=True)

CROSS_DATA_DIR = "preprocessed_data/Cross"
#EPOCHS = 100
#LEARNING_RATE = 5e-3
NUM_CLASSES = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

cross_train_loader, cross_test_loader = create_dataloaders(CROSS_DATA_DIR, BATCH_SIZE, add_person_id=True)

Loading test data...
Loading train data...
Loaded 32 training samples
Loaded 8 test samples
Class distribution in training: [8 8 8 8]
Class distribution in test: [2 2 2 2]
Train batches: 4
Test batches: 1
Using device: cuda
Loading test1 data...
Loading test2 data...
Loading test3 data...
Loading train data...
Loaded 64 training samples
Loaded 48 test samples
Class distribution in training: [16 16 16 16]
Class distribution in test: [12 12 12 12]
Train batches: 8
Test batches: 6


In [13]:
cross_dataset = cross_train_loader.dataset
intra_dataset = intra_train_loader.dataset

def build_person_subsets(dataset, source_name):
    """Build subsets of the dataset grouped by person_id."""
    if dataset.person_ids is None:
        raise ValueError(f"{source_name} dataset must be created with add_person_id=True")

    grouped_indices = {}
    for index, person_id in enumerate(dataset.person_ids):
        grouped_indices.setdefault(int(person_id), []).append(index)

    return [
        (f"{source_name}_{person_id}", Subset(dataset, indices), person_id)
        for person_id, indices in sorted(grouped_indices.items())
    ]


cross_person_splits = build_person_subsets(cross_dataset, "cross")
intra_person_splits = build_person_subsets(intra_dataset, "intra")
person_splits = cross_person_splits + intra_person_splits

print("Person-level folds:")
for split_name, subset, person_id in person_splits:
    print(f"  {split_name}: person_id={person_id}, samples={len(subset)}")

Person-level folds:
  cross_113922: person_id=113922, samples=32
  cross_164636: person_id=164636, samples=32
  intra_105923: person_id=105923, samples=32


In [14]:
cross_person_splits = build_person_subsets(cross_dataset, "cross")
intra_person_splits = build_person_subsets(intra_dataset, "intra")
person_splits = cross_person_splits + intra_person_splits

print("Person-level folds:")
for split_name, subset, person_id in person_splits:
    print(f"  {split_name}: person_id={person_id}, samples={len(subset)}")

Person-level folds:
  cross_113922: person_id=113922, samples=32
  cross_164636: person_id=164636, samples=32
  intra_105923: person_id=105923, samples=32


In [15]:
param_grid = {
    "learning_rate": [1e-3, 5e-3, 1e-2],
    "hidden_channels": [32, 64, 128],
    "kernel_size": [3, 5, 7],
    "dropout": [0.1, 0.2, 0.3],
    "batch_size": [8, 16],
}


top_models = run_person_grid_search(
    model_class=CNNBaseline1D,
    param_grid=param_grid,
    person_splits=person_splits,
    num_classes=4,
    epochs=100,
    patience=20,
)

Total configs: 162

Testing parameters:
{'learning_rate': 0.001, 'hidden_channels': 32, 'kernel_size': 3, 'dropout': 0.1, 'batch_size': 8}

Fold 1/3 | Test person: 113922
Fold 1 | Epoch 1/100 | Train Acc: 65.62% | Val Acc: 43.75% | Val Loss: 1.1330
Fold 1 | Epoch 2/100 | Train Acc: 75.00% | Val Acc: 53.12% | Val Loss: 1.0614
Fold 1 | Epoch 3/100 | Train Acc: 92.19% | Val Acc: 62.50% | Val Loss: 1.0250
Fold 1 | Epoch 4/100 | Train Acc: 96.88% | Val Acc: 75.00% | Val Loss: 0.9743
Fold 1 | Epoch 5/100 | Train Acc: 98.44% | Val Acc: 62.50% | Val Loss: 0.9886
Fold 1 | Epoch 6/100 | Train Acc: 100.00% | Val Acc: 62.50% | Val Loss: 0.9840
Fold 1 | Epoch 7/100 | Train Acc: 100.00% | Val Acc: 65.62% | Val Loss: 0.9518
Fold 1 | Epoch 8/100 | Train Acc: 98.44% | Val Acc: 75.00% | Val Loss: 0.8845
Fold 1 | Epoch 9/100 | Train Acc: 100.00% | Val Acc: 75.00% | Val Loss: 0.8133
Fold 1 | Epoch 10/100 | Train Acc: 100.00% | Val Acc: 75.00% | Val Loss: 0.7541
Fold 1 | Epoch 11/100 | Train Acc: 100.00% |

In [16]:
nr_of_runs = 25
evaluate_top_models_person_cv(CNNBaseline1D, top_models, person_splits, num_classes=4, device=DEVICE, n_runs=nr_of_runs)



MODEL 1
{'learning_rate': 0.01, 'hidden_channels': 64, 'kernel_size': 5, 'dropout': 0.2, 'batch_size': 8}

Run 1/25
  Fold 1/3 | Person 113922 | Acc 75.00%
  Fold 2/3 | Person 164636 | Acc 71.88%
  Fold 3/3 | Person 105923 | Acc 90.62%
Run 1 Mean Person-CV Accuracy: 79.17%

Run 2/25
  Fold 1/3 | Person 113922 | Acc 78.12%
  Fold 2/3 | Person 164636 | Acc 75.00%
  Fold 3/3 | Person 105923 | Acc 100.00%
Run 2 Mean Person-CV Accuracy: 84.38%

Run 3/25
  Fold 1/3 | Person 113922 | Acc 75.00%
  Fold 2/3 | Person 164636 | Acc 56.25%
  Fold 3/3 | Person 105923 | Acc 90.62%
Run 3 Mean Person-CV Accuracy: 73.96%

Run 4/25
  Fold 1/3 | Person 113922 | Acc 75.00%
  Fold 2/3 | Person 164636 | Acc 75.00%
  Fold 3/3 | Person 105923 | Acc 96.88%
Run 4 Mean Person-CV Accuracy: 82.29%

Run 5/25
  Fold 1/3 | Person 113922 | Acc 62.50%
  Fold 2/3 | Person 164636 | Acc 75.00%
  Fold 3/3 | Person 105923 | Acc 90.62%
Run 5 Mean Person-CV Accuracy: 76.04%

Run 6/25
  Fold 1/3 | Person 113922 | Acc 75.00%
  

[{'params': {'learning_rate': 0.005,
   'hidden_channels': 64,
   'kernel_size': 7,
   'dropout': 0.1,
   'batch_size': 8},
  'mean_acc': np.float64(78.16666666666666),
  'std_acc': np.float64(3.1797973380564857),
  'best_acc': np.float64(82.29166666666667),
  'worst_acc': np.float64(70.83333333333333),
  'score': np.float64(74.98686932861017)},
 {'params': {'learning_rate': 0.01,
   'hidden_channels': 64,
   'kernel_size': 3,
   'dropout': 0.1,
   'batch_size': 8},
  'mean_acc': np.float64(77.95833333333333),
  'std_acc': np.float64(3.0996191522328824),
  'best_acc': np.float64(84.375),
  'worst_acc': np.float64(72.91666666666667),
  'score': np.float64(74.85871418110045)},
 {'params': {'learning_rate': 0.01,
   'hidden_channels': 32,
   'kernel_size': 3,
   'dropout': 0.2,
   'batch_size': 8},
  'mean_acc': np.float64(79.08333333333333),
  'std_acc': np.float64(4.238137038317137),
  'best_acc': np.float64(88.54166666666667),
  'worst_acc': np.float64(69.79166666666667),
  'score': np